# 02 — Factor baselines (rungs 1-3 + 1d-3d)

Runs the 6 hand-crafted / OLS rungs of the comparison ladder, all CPU.
Each rung writes `outputs/rung_X/all_results.csv` for downstream comparison.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys; from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import config, runner


## 1. Load the canonical panel


In [ ]:
panel = pd.read_parquet(config.PANEL_DAILY_PARQUET)
panel['date'] = pd.to_datetime(panel['date'])
print(f'panel: {len(panel):,} rows, {panel["date"].min()} → {panel["date"].max()}')


## 2. Run rungs 1, 1d, 2, 2d, 3, 3d via runner


In [ ]:
BASELINE_CONFIGS = [
    'rung_1_simple_momentum_decile.yaml',
    'rung_1d_simple_momentum_dual.yaml',
    'rung_2_ewm_momentum_decile.yaml',
    'rung_2d_ewm_momentum_dual.yaml',
    'rung_3_ts_regression_decile.yaml',
    'rung_3d_ts_regression_dual.yaml',
]
configs = [runner.load_config(config.EXPERIMENTS_DIR / c) for c in BASELINE_CONFIGS]
runner.status_table(configs)


In [ ]:
# Run any cells not already complete (resumable)
for cfg in configs:
    runner.run_experiment(cfg, panel)


## 3. Load results + compare


In [ ]:
from src import eval as evalmod
rung_ids = [c['experiment_id'] for c in configs]
master = evalmod.load_master_results(rung_ids)
master.head()


In [ ]:
# Sharpe + max-DD per rung (averaged across seeds for trainable rungs; here all are non-trainable)
summary = (master.groupby('experiment_id')['return']
                  .apply(lambda s: pd.Series(evalmod.perf_summary(s)))
                  .unstack())
summary[['ann_return', 'ann_vol', 'sharpe', 'max_dd']].round(3)


## 4. Side-by-side cumulative P&L (cumsum, not cumprod — convention)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for exp_id in rung_ids:
    df = master[master['experiment_id'] == exp_id].sort_values('date')
    ax.plot(df['date'], df['return'].cumsum(), label=exp_id, alpha=0.85)
ax.set(title='Cumulative P&L (cumsum) — baseline rungs', ylabel='cum return')
ax.legend(loc='upper left', fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
